In [1]:
import pandas as pd
import numpy as np

In [2]:
path_folder = "Datasets/"
path_claim = "ClaimDetails_for_distribution.xlsx"

In [3]:
ds_claim = pd.read_excel(
    path_folder+path_claim,
    sheet_name=0
)

In [4]:
ds_claim = ds_claim.drop(columns=[
    "Claim Number",
    "Weather Indicator",
    "CAT Code",
    "Loss Date",
    "NOL Date",
    "Claimant Number",
    "Weather",
    "Loss Type",
    "Accident City",
    "Accident Zip"
])
ds_claim.columns

Index(['Territory', 'CAT Severity Code', 'Peril Description',
       'Cause Of Injury Text', 'Peril Group', 'Weather Text', 'Division',
       'Accident State', 'Policy Form Desc', '6_cluster_cluster_id',
       '6_cluster_distance_to_centroid_km', '7_cluster_cluster_id',
       '7_cluster_distance_to_centroid_km', '8_cluster_cluster_id',
       '8_cluster_distance_to_centroid_km', '9_cluster_cluster_id',
       '9_cluster_distance_to_centroid_km', '10_cluster_cluster_id',
       '10_cluster_distance_to_centroid_km'],
      dtype='object')

In [5]:
# =========================================================
# GLOBAL EXPERIMENT METRICS TABLE
# =========================================================

metrics_table = pd.DataFrame(columns=[
    "Experiment",
    "Model",
    "TextStrategy",
    "Accuracy",
    "MAE",
    "QWK"
])
# =========================================================
# FUNCTION TO ADD RESULTS
# =========================================================

def add_experiment_result(
        metrics_table,
        experiment_name,
        model,
        text_strategy,
        accuracy,
        mae,
        qwk):

    new_row = pd.DataFrame([{
        "Experiment": experiment_name,
        "Model": model,
        "TextStrategy": text_strategy,
        "Accuracy": accuracy,
        "MAE": mae,
        "QWK": qwk
    }])

    metrics_table = pd.concat([metrics_table, new_row], ignore_index=True)

    return metrics_table

In [6]:
# =========================================================
# BEST MODEL SO FAR (CV + OOF + threshold optimization)
# - LinearSVR regression (C=1.0)
# - Word TF-IDF (1,2), max_features=8000, min_df=2
# - Categorical OHE + numeric features
# - Optimize cutpoints on OOF continuous predictions to maximize QWK
#
# Outputs:
#   - metrics_table_base: fold metrics + OOF_round row
#   - comparison_table: naive_round vs optimized_thresholds metrics
#   - best_thresholds: (t1,t2,t3,t4)
#   - oof_detail: per-row y_true, y_pred_continuous, y_pred_round, y_pred_threshopt
#   - confusion matrices for both mappings
# =========================================================

import numpy as np
import pandas as pd
from itertools import product

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, mean_absolute_error, cohen_kappa_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.svm import LinearSVR


# =========================
# 1) LOAD DATA
# =========================
ds_claim = pd.read_excel(path_folder + path_claim, sheet_name=0).copy()

# =========================
# 2) CONFIG
# =========================
TARGET_COL = "CAT Severity Code"

TEXT_COLS = ["Cause Of Injury Text"]

CATEGORICAL_COLS = [
    "Division", 
    "Weather Text", 
    "Peril Description"
]

NUMERIC_COLS = [
]

N_SPLITS = 5
RANDOM_STATE = 42

TFIDF_PARAMS = {
    "analyzer": "word",
    "ngram_range": (1, 4),
    "max_features": None,
    "min_df": 2,
    "sublinear_tf": True,
}

BASE_REGRESSOR = LinearSVR(C=1.0, epsilon=0.0, max_iter=10000, random_state=RANDOM_STATE)

EXPERIMENT_NAME_BASE = "model_F_base"
EXPERIMENT_NAME_THRESH = "model_F_thresh"


# =========================
# 3) BASIC CLEANING
# =========================
df = ds_claim.copy()

# Keep only labeled rows for this evaluation experiment
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
df = df[df[TARGET_COL].isin([1, 2, 3, 4, 5])].copy()
df[TARGET_COL] = df[TARGET_COL].astype(int)

# Strip whitespace in object columns (important for Territory / Policy Form strings)
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].astype(str).str.strip()
        df.loc[df[col].isin(["nan", "None", ""]), col] = np.nan

# Ensure categoricals are clean strings
for col in CATEGORICAL_COLS:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()
        df.loc[df[col].isin(["nan", "None", ""]), col] = np.nan

# Numeric coercion
for col in NUMERIC_COLS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")


# =========================
# 4) COMBINE TEXT (Option A)
# =========================
def combine_text_columns(dataframe, text_cols, new_col="combined_text"):
    temp = dataframe.copy()
    for c in text_cols:
        temp[c] = temp[c].fillna("").astype(str).str.strip()
    temp[new_col] = (
        temp[text_cols]
        .agg(" ".join, axis=1)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )
    return temp


FEATURE_COLS = TEXT_COLS + CATEGORICAL_COLS + NUMERIC_COLS
df_model = df[FEATURE_COLS + [TARGET_COL]].copy()


# =========================
# 5) PREPROCESSOR
# =========================
text_transformer = Pipeline([
    ("selector", FunctionTransformer(lambda x: x.squeeze(), validate=False)),
    ("tfidf", TfidfVectorizer(**TFIDF_PARAMS)),
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])


text_cause = Pipeline([
    ("selector", FunctionTransformer(lambda x: x.squeeze(), validate=False)),
    ("tfidf", TfidfVectorizer(**TFIDF_PARAMS))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("cause_text", TfidfVectorizer(**TFIDF_PARAMS), "Cause Of Injury Text"),
        ("cat", categorical_transformer, CATEGORICAL_COLS),
    ],
    remainder="drop"
)


# =========================
# 6) HELPERS
# =========================
def clip_predictions(pred_continuous, lo=1.0, hi=5.0):
    return np.clip(pred_continuous, lo, hi)

def round_predictions(pred_continuous):
    return np.rint(clip_predictions(pred_continuous)).astype(int)

def apply_thresholds(pred_continuous, thresholds):
    """
    thresholds = (t1, t2, t3, t4), strictly increasing
    """
    pred = clip_predictions(pred_continuous)
    t1, t2, t3, t4 = thresholds
    return np.where(
        pred < t1, 1,
        np.where(pred < t2, 2,
        np.where(pred < t3, 3,
        np.where(pred < t4, 4, 5)))
    ).astype(int)

def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "mae": mean_absolute_error(y_true, y_pred),
        "qwk": cohen_kappa_score(y_true, y_pred, weights="quadratic"),
    }

def optimize_thresholds_grid(y_true, pred_continuous):
    """
    Grid-search cutpoints to maximize QWK on provided predictions.
    NOTE: This uses the same predictions for tuning+evaluation (useful for model selection).
    """
    best = {"qwk": -np.inf, "thresholds": None, "metrics": None}

    grid_t1 = np.arange(1.2, 2.21, 0.1)
    grid_t2 = np.arange(2.0, 3.21, 0.1)
    grid_t3 = np.arange(2.8, 4.21, 0.1)
    grid_t4 = np.arange(3.6, 4.81, 0.1)

    for t1, t2, t3, t4 in product(grid_t1, grid_t2, grid_t3, grid_t4):
        if not (t1 < t2 < t3 < t4):
            continue
        y_pred = apply_thresholds(pred_continuous, (t1, t2, t3, t4))
        m = compute_metrics(y_true, y_pred)
        if m["qwk"] > best["qwk"]:
            best = {"qwk": m["qwk"], "thresholds": (float(t1), float(t2), float(t3), float(t4)), "metrics": m}

    return best["thresholds"], best["metrics"]


# =========================
# 7) CV TO GET OOF CONTINUOUS PREDICTIONS
# =========================
def run_cv_oof_continuous(df_input, target_col, feature_cols, preprocessor, base_regressor,
                         n_splits=5, random_state=42, experiment_name="experiment"):
    X = df_input[feature_cols].copy()
    y = df_input[target_col].astype(int).values

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    oof_pred_cont = np.zeros(len(df_input), dtype=float)
    oof_pred_round = np.zeros(len(df_input), dtype=int)
    fold_rows = []

    for fold, (tr, va) in enumerate(skf.split(X, y), start=1):
        X_tr, X_va = X.iloc[tr].copy(), X.iloc[va].copy()
        y_tr, y_va = y[tr], y[va]

        X_tr_t = preprocessor.fit_transform(X_tr)
        X_va_t = preprocessor.transform(X_va)

        reg = clone(base_regressor)
        reg.fit(X_tr_t, y_tr)

        va_cont = clip_predictions(reg.predict(X_va_t))
        va_round = round_predictions(va_cont)

        oof_pred_cont[va] = va_cont
        oof_pred_round[va] = va_round

        m = compute_metrics(y_va, va_round)
        fold_rows.append({
            "experiment": experiment_name,
            "fold": fold,
            "n_train": len(tr),
            "n_valid": len(va),
            **m
        })

        print(f"{experiment_name} | Fold {fold}: accuracy={m['accuracy']:.4f} | mae={m['mae']:.4f} | qwk={m['qwk']:.4f}")

    oof_m = compute_metrics(y, oof_pred_round)
    fold_rows.append({
        "experiment": experiment_name,
        "fold": "OOF_round",
        "n_train": None,
        "n_valid": len(df_input),
        **oof_m
    })

    metrics_table = pd.DataFrame(fold_rows)

    oof_detail = df_input.copy()
    oof_detail["y_true"] = y
    oof_detail["y_pred_continuous"] = oof_pred_cont
    oof_detail["y_pred_round"] = oof_pred_round

    return metrics_table, oof_detail


# =========================
# 8) RUN BEST MODEL EXPERIMENT
# =========================
metrics_table_base, oof_detail = run_cv_oof_continuous(
    df_input=df_model,
    target_col=TARGET_COL,
    feature_cols=FEATURE_COLS,
    preprocessor=preprocessor,
    base_regressor=BASE_REGRESSOR,
    n_splits=N_SPLITS,
    random_state=RANDOM_STATE,
    experiment_name=EXPERIMENT_NAME_BASE
)

print("\nBase metrics table (includes OOF_round row):")
print(metrics_table_base)

y_true = oof_detail["y_true"].values
pred_cont = oof_detail["y_pred_continuous"].values
pred_round = oof_detail["y_pred_round"].values

base_metrics = compute_metrics(y_true, pred_round)
base_cm = confusion_matrix(y_true, pred_round, labels=[1,2,3,4,5])

print("\nBaseline (naive rounding) OOF metrics:")
print(base_metrics)
print("\nBaseline confusion matrix (rows=true, cols=pred):")
print(pd.DataFrame(base_cm, index=[1,2,3,4,5], columns=[1,2,3,4,5]))

# Threshold optimization on OOF preds (model-selection step)
best_thresholds, best_thresh_metrics = optimize_thresholds_grid(y_true, pred_cont)
pred_thresh = apply_thresholds(pred_cont, best_thresholds)
thresh_cm = confusion_matrix(y_true, pred_thresh, labels=[1,2,3,4,5])

print("\nBest optimized thresholds found:")
print(best_thresholds)
print("\nOptimized-threshold OOF metrics:")
print(best_thresh_metrics)
print("\nOptimized-threshold confusion matrix (rows=true, cols=pred):")
print(pd.DataFrame(thresh_cm, index=[1,2,3,4,5], columns=[1,2,3,4,5]))

comparison_table = pd.DataFrame([
    {"experiment": EXPERIMENT_NAME_BASE,  "mapping": "naive_round",          "thresholds": (1.5,2.5,3.5,4.5),  **base_metrics},
    {"experiment": EXPERIMENT_NAME_THRESH,"mapping": "optimized_thresholds", "thresholds": best_thresholds,      **best_thresh_metrics},
])

print("\nComparison table:")
print(comparison_table)

# Add to oof_detail for inspection (in-memory only)
oof_detail["y_pred_threshopt"] = pred_thresh
oof_detail["thresholds_used"] = str(best_thresholds)

# If you want these objects in your notebook:
# metrics_table_base, comparison_table, best_thresholds, oof_detail

metrics_table = add_experiment_result(
    metrics_table,
    experiment_name="Model F",
    model="LinearSVR",
    text_strategy="Model E + ",
    accuracy=best_thresh_metrics["accuracy"],
    mae=best_thresh_metrics["mae"],
    qwk=best_thresh_metrics["qwk"]
)

print(metrics_table.sort_values("QWK", ascending=False))

model_F_base | Fold 1: accuracy=0.4843 | mae=0.6478 | qwk=0.5256
model_F_base | Fold 2: accuracy=0.4873 | mae=0.6962 | qwk=0.4475
model_F_base | Fold 3: accuracy=0.4873 | mae=0.6962 | qwk=0.4589
model_F_base | Fold 4: accuracy=0.4430 | mae=0.7152 | qwk=0.4873
model_F_base | Fold 5: accuracy=0.4810 | mae=0.7152 | qwk=0.4391

Base metrics table (includes OOF_round row):
     experiment       fold  n_train  n_valid  accuracy       mae       qwk
0  model_F_base          1    632.0      159  0.484277  0.647799  0.525639
1  model_F_base          2    633.0      158  0.487342  0.696203  0.447529
2  model_F_base          3    633.0      158  0.487342  0.696203  0.458884
3  model_F_base          4    633.0      158  0.443038  0.715190  0.487275
4  model_F_base          5    633.0      158  0.481013  0.715190  0.439101
5  model_F_base  OOF_round      NaN      791  0.476612  0.694058  0.471723

Baseline (naive rounding) OOF metrics:
{'accuracy': 0.47661188369152974, 'mae': 0.6940581542351454, 'qw

/tmp/ipykernel_3433719/2035950194.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  metrics_table = pd.concat([metrics_table, new_row], ignore_index=True)


In [7]:
# =========================================================
# BEST MODEL SO FAR (CV + OOF + threshold optimization)
# - LinearSVR regression (C=1.0)
# - Word TF-IDF (1,2), max_features=8000, min_df=2
# - Categorical OHE + numeric features
# - Optimize cutpoints on OOF continuous predictions to maximize QWK
#
# Outputs:
#   - metrics_table_base: fold metrics + OOF_round row
#   - comparison_table: naive_round vs optimized_thresholds metrics
#   - best_thresholds: (t1,t2,t3,t4)
#   - oof_detail: per-row y_true, y_pred_continuous, y_pred_round, y_pred_threshopt
#   - confusion matrices for both mappings
# =========================================================

import numpy as np
import pandas as pd
from itertools import product

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, mean_absolute_error, cohen_kappa_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.svm import LinearSVR


# =========================
# 1) LOAD DATA
# =========================
ds_claim = pd.read_excel(path_folder + path_claim, sheet_name=0).copy()

# =========================
# 2) CONFIG
# =========================
TARGET_COL = "CAT Severity Code"

TEXT_COLS = ["Cause Of Injury Text"]

CATEGORICAL_COLS = [
    "Division", 
    "Weather Text", 
    "Peril Description"
]

NUMERIC_COLS = [
]

N_SPLITS = 5
RANDOM_STATE = 42

TFIDF_PARAMS = {
    "analyzer": "word",
    "ngram_range": (1, 4),
    "max_features": None,
    "min_df": 2,
    "sublinear_tf": True,
}

BASE_REGRESSOR = LinearSVR(C=1.0, epsilon=0.0, max_iter=10000, random_state=RANDOM_STATE)

EXPERIMENT_NAME_BASE = "model_J_base"
EXPERIMENT_NAME_THRESH = "model_J_tresh"


# =========================
# 3) BASIC CLEANING
# =========================
df = ds_claim.copy()

# Keep only labeled rows for this evaluation experiment
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
df = df[df[TARGET_COL].isin([1, 2, 3, 4, 5])].copy()
df[TARGET_COL] = df[TARGET_COL].astype(int)

# NEW: target transformation
df["target_transformed"] = df[TARGET_COL] - 3

# Strip whitespace in object columns (important for Territory / Policy Form strings)
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].astype(str).str.strip()
        df.loc[df[col].isin(["nan", "None", ""]), col] = np.nan

# Ensure categoricals are clean strings
for col in CATEGORICAL_COLS:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()
        df.loc[df[col].isin(["nan", "None", ""]), col] = np.nan

# Numeric coercion
for col in NUMERIC_COLS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")


# =========================
# 4) COMBINE TEXT (Option A)
# =========================
def combine_text_columns(dataframe, text_cols, new_col="combined_text"):
    temp = dataframe.copy()
    for c in text_cols:
        temp[c] = temp[c].fillna("").astype(str).str.strip()
    temp[new_col] = (
        temp[text_cols]
        .agg(" ".join, axis=1)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )
    return temp


FEATURE_COLS = TEXT_COLS + CATEGORICAL_COLS + NUMERIC_COLS
df_model = df[FEATURE_COLS + [TARGET_COL, "target_transformed"]].copy()


# =========================
# 5) PREPROCESSOR
# =========================
text_transformer = Pipeline([
    ("selector", FunctionTransformer(lambda x: x.squeeze(), validate=False)),
    ("tfidf", TfidfVectorizer(**TFIDF_PARAMS)),
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])


text_cause = Pipeline([
    ("selector", FunctionTransformer(lambda x: x.squeeze(), validate=False)),
    ("tfidf", TfidfVectorizer(**TFIDF_PARAMS))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("cause_text", TfidfVectorizer(**TFIDF_PARAMS), "Cause Of Injury Text"),
        ("cat", categorical_transformer, CATEGORICAL_COLS),
    ],
    remainder="drop"
)


# =========================
# 6) HELPERS
# =========================
def clip_predictions(pred_continuous, lo=1.0, hi=5.0):
    return np.clip(pred_continuous, lo, hi)

def round_predictions(pred_continuous):
    return np.rint(clip_predictions(pred_continuous)).astype(int)

def apply_thresholds(pred_continuous, thresholds):
    """
    thresholds = (t1, t2, t3, t4), strictly increasing
    """
    pred = clip_predictions(pred_continuous)
    t1, t2, t3, t4 = thresholds
    return np.where(
        pred < t1, 1,
        np.where(pred < t2, 2,
        np.where(pred < t3, 3,
        np.where(pred < t4, 4, 5)))
    ).astype(int)

def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "mae": mean_absolute_error(y_true, y_pred),
        "qwk": cohen_kappa_score(y_true, y_pred, weights="quadratic"),
    }

def optimize_thresholds_grid(y_true, pred_continuous):
    """
    Grid-search cutpoints to maximize QWK on provided predictions.
    NOTE: This uses the same predictions for tuning+evaluation (useful for model selection).
    """
    best = {"qwk": -np.inf, "thresholds": None, "metrics": None}

    grid_t1 = np.arange(1.2, 2.21, 0.1)
    grid_t2 = np.arange(2.0, 3.21, 0.1)
    grid_t3 = np.arange(2.8, 4.21, 0.1)
    grid_t4 = np.arange(3.6, 4.81, 0.1)

    for t1, t2, t3, t4 in product(grid_t1, grid_t2, grid_t3, grid_t4):
        if not (t1 < t2 < t3 < t4):
            continue
        y_pred = apply_thresholds(pred_continuous, (t1, t2, t3, t4))
        m = compute_metrics(y_true, y_pred)
        if m["qwk"] > best["qwk"]:
            best = {"qwk": m["qwk"], "thresholds": (float(t1), float(t2), float(t3), float(t4)), "metrics": m}

    return best["thresholds"], best["metrics"]


# =========================
# 7) CV TO GET OOF CONTINUOUS PREDICTIONS
# =========================
def run_cv_oof_continuous(df_input, target_col, feature_cols, preprocessor, base_regressor,
                         n_splits=5, random_state=42, experiment_name="experiment"):
    X = df_input[feature_cols].copy()

    # Original target for evaluation
    y_true_orig = df_input[target_col].astype(int).values

    # Transformed target for training
    y_train_trans = df_input["target_transformed"].values

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    oof_pred_cont = np.zeros(len(df_input), dtype=float)
    oof_pred_round = np.zeros(len(df_input), dtype=int)
    fold_rows = []

    for fold, (tr, va) in enumerate(skf.split(X, y_true_orig), start=1):
        X_tr, X_va = X.iloc[tr].copy(), X.iloc[va].copy()

        y_tr_trans = y_train_trans[tr]
        y_va_true = y_true_orig[va]

        X_tr_t = preprocessor.fit_transform(X_tr)
        X_va_t = preprocessor.transform(X_va)

        reg = clone(base_regressor)
        reg.fit(X_tr_t, y_tr_trans)

        va_cont = reg.predict(X_va_t)

        # reverse transform back to original class scale
        va_cont = va_cont + 3
        va_cont = clip_predictions(va_cont)
        va_round = round_predictions(va_cont)

        oof_pred_cont[va] = va_cont
        oof_pred_round[va] = va_round

        m = compute_metrics(y_va_true, va_round)
        fold_rows.append({
            "experiment": experiment_name,
            "fold": fold,
            "n_train": len(tr),
            "n_valid": len(va),
            **m
        })

        print(f"{experiment_name} | Fold {fold}: accuracy={m['accuracy']:.4f} | mae={m['mae']:.4f} | qwk={m['qwk']:.4f}")

    oof_m = compute_metrics(y_true_orig, oof_pred_round)
    fold_rows.append({
        "experiment": experiment_name,
        "fold": "OOF_round",
        "n_train": None,
        "n_valid": len(df_input),
        **oof_m
    })

    metrics_table = pd.DataFrame(fold_rows)

    oof_detail = df_input.copy()
    oof_detail["y_true"] = y_true_orig
    oof_detail["y_pred_continuous"] = oof_pred_cont
    oof_detail["y_pred_round"] = oof_pred_round

    return metrics_table, oof_detail


# =========================
# 8) RUN BEST MODEL EXPERIMENT
# =========================
metrics_table_base, oof_detail = run_cv_oof_continuous(
    df_input=df_model,
    target_col=TARGET_COL,
    feature_cols=FEATURE_COLS,
    preprocessor=preprocessor,
    base_regressor=BASE_REGRESSOR,
    n_splits=N_SPLITS,
    random_state=RANDOM_STATE,
    experiment_name=EXPERIMENT_NAME_BASE
)

print("\nBase metrics table (includes OOF_round row):")
print(metrics_table_base)

y_true = oof_detail["y_true"].values
pred_cont = oof_detail["y_pred_continuous"].values
pred_round = oof_detail["y_pred_round"].values

base_metrics = compute_metrics(y_true, pred_round)
base_cm = confusion_matrix(y_true, pred_round, labels=[1,2,3,4,5])

print("\nBaseline (naive rounding) OOF metrics:")
print(base_metrics)
print("\nBaseline confusion matrix (rows=true, cols=pred):")
print(pd.DataFrame(base_cm, index=[1,2,3,4,5], columns=[1,2,3,4,5]))

# Threshold optimization on OOF preds (model-selection step)
best_thresholds, best_thresh_metrics = optimize_thresholds_grid(y_true, pred_cont)
pred_thresh = apply_thresholds(pred_cont, best_thresholds)
thresh_cm = confusion_matrix(y_true, pred_thresh, labels=[1,2,3,4,5])

print("\nBest optimized thresholds found:")
print(best_thresholds)
print("\nOptimized-threshold OOF metrics:")
print(best_thresh_metrics)
print("\nOptimized-threshold confusion matrix (rows=true, cols=pred):")
print(pd.DataFrame(thresh_cm, index=[1,2,3,4,5], columns=[1,2,3,4,5]))

comparison_table = pd.DataFrame([
    {"experiment": EXPERIMENT_NAME_BASE,  "mapping": "naive_round",          "thresholds": (1.5,2.5,3.5,4.5),  **base_metrics},
    {"experiment": EXPERIMENT_NAME_THRESH,"mapping": "optimized_thresholds", "thresholds": best_thresholds,      **best_thresh_metrics},
])

print("\nComparison table:")
print(comparison_table)

# Add to oof_detail for inspection (in-memory only)
oof_detail["y_pred_threshopt"] = pred_thresh
oof_detail["thresholds_used"] = str(best_thresholds)

# If you want these objects in your notebook:
# metrics_table_base, comparison_table, best_thresholds, oof_detail

metrics_table = add_experiment_result(
    metrics_table,
    experiment_name="Model J",
    model="LinearSVR",
    text_strategy="Model F + ordinal structure modeling",
    accuracy=best_thresh_metrics["accuracy"],
    mae=best_thresh_metrics["mae"],
    qwk=best_thresh_metrics["qwk"]
)

print(metrics_table.sort_values("QWK", ascending=False))

model_J_base | Fold 1: accuracy=0.4906 | mae=0.6541 | qwk=0.5094
model_J_base | Fold 2: accuracy=0.4810 | mae=0.7025 | qwk=0.4411
model_J_base | Fold 3: accuracy=0.4873 | mae=0.6962 | qwk=0.4589
model_J_base | Fold 4: accuracy=0.4557 | mae=0.7025 | qwk=0.4958
model_J_base | Fold 5: accuracy=0.4810 | mae=0.7152 | qwk=0.4460

Base metrics table (includes OOF_round row):
     experiment       fold  n_train  n_valid  accuracy       mae       qwk
0  model_J_base          1    632.0      159  0.490566  0.654088  0.509377
1  model_J_base          2    633.0      158  0.481013  0.702532  0.441077
2  model_J_base          3    633.0      158  0.487342  0.696203  0.458884
3  model_J_base          4    633.0      158  0.455696  0.702532  0.495751
4  model_J_base          5    633.0      158  0.481013  0.715190  0.445968
5  model_J_base  OOF_round      NaN      791  0.479140  0.694058  0.470367

Baseline (naive rounding) OOF metrics:
{'accuracy': 0.47914032869785084, 'mae': 0.6940581542351454, 'qw

In [8]:
# =========================================================
# MODEL L
# Best TF-IDF setup + semantic extracted tags + structured categoricals
# + LinearSVR + threshold optimization
#
# Rationale:
#   - Keep the strong raw lexical signal from TF-IDF
#   - Add semantic rule-based tags as structured auxiliary features
#   - Keep existing structured categorical variables
# =========================================================

import re
import numpy as np
import pandas as pd
from itertools import product

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, mean_absolute_error, cohen_kappa_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.svm import LinearSVR


# =========================
# 1) LOAD DATA
# =========================
ds_claim = pd.read_excel(path_folder + path_claim, sheet_name=0).copy()


# =========================
# 2) CONFIG
# =========================
TARGET_COL = "CAT Severity Code"
RAW_TEXT_COL = "Cause Of Injury Text"

# Keep the strong structured columns you already validated
BASE_CATEGORICAL_COLS = [
    "Division",
    "Weather Text",
    "Peril Description"
]

NUMERIC_COLS = []

N_SPLITS = 5
RANDOM_STATE = 42

# Use your strongest word-TFIDF-style setup as baseline
TFIDF_PARAMS = {
    "analyzer": "word",
    "ngram_range": (1, 4),
    "max_features": None,
    "min_df": 2,
    "sublinear_tf": True,
}

BASE_REGRESSOR = LinearSVR(
    C=1.0,
    epsilon=0.0,
    max_iter=10000,
    random_state=RANDOM_STATE
)

EXPERIMENT_NAME_BASE = "model_L_base"
EXPERIMENT_NAME_THRESH = "model_L_tresh"


# =========================
# 3) BASIC CLEANING
# =========================
df = ds_claim.copy()

df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
df = df[df[TARGET_COL].isin([1, 2, 3, 4, 5])].copy()
df[TARGET_COL] = df[TARGET_COL].astype(int)

for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].astype(str).str.strip()
        df.loc[df[col].isin(["nan", "None", ""]), col] = np.nan

for col in BASE_CATEGORICAL_COLS:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()
        df.loc[df[col].isin(["nan", "None", ""]), col] = np.nan

for col in NUMERIC_COLS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df[RAW_TEXT_COL] = df[RAW_TEXT_COL].fillna("").astype(str).str.strip()


# =========================
# 4) RULE-BASED TAG EXTRACTION
# =========================
def normalize_text(text: str) -> str:
    text = str(text).lower().strip()
    text = re.sub(r"[^a-z0-9\s\/\-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def has_any(text: str, patterns) -> bool:
    return any(p in text for p in patterns)

def extract_event_family(text: str) -> str:
    if has_any(text, ["tornado", "hurricane", "storm", "wind", "gust", "shingle blown", "tree fell", "tree fall"]):
        return "wind"
    if has_any(text, ["hail", "ice dam", "ice "]):
        return "hail_ice"
    if has_any(text, ["water", "flood", "seepage", "leak", "overflow", "backup", "back-up", "sump", "sewer"]):
        return "water"
    if has_any(text, ["fire", "smoke", "burn", "char"]):
        return "fire"
    if has_any(text, ["theft", "stolen", "burglary", "robbery"]):
        return "theft"
    if has_any(text, ["collision", "crash", "struck", "hit", "rear end", "rear-end", "vehicle"]):
        return "collision_vehicle"
    if has_any(text, ["fall", "fell", "slip", "trip"]):
        return "fall_injury"
    if has_any(text, ["lightning"]):
        return "lightning"
    return "other"

def extract_damage_object(text: str) -> str:
    if has_any(text, ["roof", "shingle", "flashing", "attic"]):
        return "roof"
    if has_any(text, ["basement", "crawlspace", "crawl space", "foundation"]):
        return "basement_foundation"
    if has_any(text, ["window", "glass", "door", "garage door"]):
        return "window_door"
    if has_any(text, ["siding", "gutter", "fence", "exterior", "deck"]):
        return "exterior_structure"
    if has_any(text, ["ceiling", "wall", "floor", "drywall", "carpet", "kitchen", "bathroom", "interior"]):
        return "interior"
    if has_any(text, ["vehicle", "car", "truck", "auto"]):
        return "vehicle"
    if has_any(text, ["body", "injury", "claimant", "person", "medical"]):
        return "bodily_injury"
    return "other"

def extract_mechanism(text: str) -> str:
    if has_any(text, ["leak", "leaking", "roof leak", "flashing leak"]):
        return "leak"
    if has_any(text, ["seepage", "seep"]):
        return "seepage"
    if has_any(text, ["overflow", "backup", "back-up", "sump", "sewer"]):
        return "backup_overflow"
    if has_any(text, ["fell", "fall", "tree fell", "collapse", "collapsed"]):
        return "collapse_fall"
    if has_any(text, ["impact", "struck", "hit", "blown into"]):
        return "impact"
    if has_any(text, ["crack", "broken", "broke", "damage to"]):
        return "breakage"
    return "unknown"

def extract_catastrophe_flag(text: str, weather_text: str, peril_desc: str) -> str:
    combo = f"{text} {weather_text} {peril_desc}".lower()
    if has_any(combo, ["tornado", "hurricane", "tropical storm", "cat", "catastrophe"]):
        return "yes"
    return "no"

def extract_injury_flag(text: str, division: str) -> str:
    combo = f"{text} {division}".lower()
    if has_any(combo, ["injury", "injured", "claimant", "medical", "bodily", "slip", "trip", "fall"]):
        return "yes"
    if "pi" in combo or "personal" in combo:
        return "yes"
    return "no"

def extract_severity_cue(text: str) -> str:
    if has_any(text, ["total loss", "destroyed", "major", "severe", "extensive", "multiple", "entire", "collapsed"]):
        return "severe"
    if has_any(text, ["moderate", "significant", "substantial"]):
        return "moderate"
    if has_any(text, ["minor", "small", "slight", "limited"]):
        return "minor"
    return "unknown"

def extract_weather_specific(text: str, weather_text: str, peril_desc: str) -> str:
    combo = f"{text} {weather_text} {peril_desc}".lower()
    if "tornado" in combo:
        return "tornado"
    if "hurricane" in combo:
        return "hurricane"
    if "tropical storm" in combo:
        return "tropical_storm"
    if "hail" in combo:
        return "hail"
    if "wind" in combo or "storm" in combo:
        return "wind_storm"
    if "flood" in combo:
        return "flood"
    return "none"

def extract_location_context(text: str) -> str:
    if has_any(text, ["basement", "crawlspace", "crawl space"]):
        return "basement"
    if has_any(text, ["roof", "attic", "ceiling"]):
        return "roof_upper"
    if has_any(text, ["interior", "inside", "kitchen", "bathroom", "bedroom", "living room", "wall", "floor"]):
        return "inside"
    if has_any(text, ["outside", "exterior", "yard", "driveway", "fence", "siding"]):
        return "outside"
    return "unknown"

def build_semantic_tags(dataframe: pd.DataFrame) -> pd.DataFrame:
    temp = dataframe.copy()

    norm_text = temp[RAW_TEXT_COL].fillna("").astype(str).map(normalize_text)
    weather_text = temp["Weather Text"].fillna("").astype(str).map(normalize_text) if "Weather Text" in temp.columns else ""
    peril_desc = temp["Peril Description"].fillna("").astype(str).map(normalize_text) if "Peril Description" in temp.columns else ""
    division = temp["Division"].fillna("").astype(str).map(normalize_text) if "Division" in temp.columns else ""

    temp["tag_event_family"] = norm_text.map(extract_event_family)
    temp["tag_damage_object"] = norm_text.map(extract_damage_object)
    temp["tag_mechanism"] = norm_text.map(extract_mechanism)
    temp["tag_catastrophe_flag"] = [
        extract_catastrophe_flag(t, w, p) for t, w, p in zip(norm_text, weather_text, peril_desc)
    ]
    temp["tag_injury_flag"] = [
        extract_injury_flag(t, d) for t, d in zip(norm_text, division)
    ]
    temp["tag_severity_cue"] = norm_text.map(extract_severity_cue)
    temp["tag_weather_specific"] = [
        extract_weather_specific(t, w, p) for t, w, p in zip(norm_text, weather_text, peril_desc)
    ]
    temp["tag_location_context"] = norm_text.map(extract_location_context)

    return temp

df = build_semantic_tags(df)

SEMANTIC_TAG_COLS = [
    "tag_event_family",
    "tag_damage_object",
    "tag_mechanism",
    "tag_catastrophe_flag",
    "tag_injury_flag",
    "tag_severity_cue",
    "tag_weather_specific",
    "tag_location_context",
]


# =========================
# 5) FEATURES
# =========================
FEATURE_COLS = [RAW_TEXT_COL] + BASE_CATEGORICAL_COLS + SEMANTIC_TAG_COLS + NUMERIC_COLS
df_model = df[FEATURE_COLS + [TARGET_COL]].copy()


# =========================
# 6) PREPROCESSOR
# =========================
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

transformers = [
    ("text", TfidfVectorizer(**TFIDF_PARAMS), RAW_TEXT_COL),
    ("base_cat", categorical_transformer, BASE_CATEGORICAL_COLS),
    ("semantic_tags", categorical_transformer, SEMANTIC_TAG_COLS),
]

if len(NUMERIC_COLS) > 0:
    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    transformers.append(("num", numeric_transformer, NUMERIC_COLS))

preprocessor = ColumnTransformer(
    transformers=transformers,
    remainder="drop"
)


# =========================
# 7) HELPERS
# =========================
def clip_predictions(pred_continuous, lo=1.0, hi=5.0):
    return np.clip(pred_continuous, lo, hi)

def round_predictions(pred_continuous):
    return np.rint(clip_predictions(pred_continuous)).astype(int)

def apply_thresholds(pred_continuous, thresholds):
    pred = clip_predictions(pred_continuous)
    t1, t2, t3, t4 = thresholds
    return np.where(
        pred < t1, 1,
        np.where(pred < t2, 2,
        np.where(pred < t3, 3,
        np.where(pred < t4, 4, 5)))
    ).astype(int)

def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "mae": mean_absolute_error(y_true, y_pred),
        "qwk": cohen_kappa_score(y_true, y_pred, weights="quadratic"),
    }

def optimize_thresholds_grid(y_true, pred_continuous):
    best = {"qwk": -np.inf, "thresholds": None, "metrics": None}

    grid_t1 = np.arange(1.2, 2.21, 0.1)
    grid_t2 = np.arange(2.0, 3.21, 0.1)
    grid_t3 = np.arange(2.8, 4.21, 0.1)
    grid_t4 = np.arange(3.6, 4.81, 0.1)

    for t1, t2, t3, t4 in product(grid_t1, grid_t2, grid_t3, grid_t4):
        if not (t1 < t2 < t3 < t4):
            continue

        y_pred = apply_thresholds(pred_continuous, (t1, t2, t3, t4))
        m = compute_metrics(y_true, y_pred)

        if m["qwk"] > best["qwk"]:
            best = {
                "qwk": m["qwk"],
                "thresholds": (float(t1), float(t2), float(t3), float(t4)),
                "metrics": m
            }

    return best["thresholds"], best["metrics"]


# =========================
# 8) CV TO GET OOF CONTINUOUS PREDICTIONS
# =========================
def run_cv_oof_continuous(
    df_input,
    target_col,
    feature_cols,
    preprocessor,
    base_regressor,
    n_splits=5,
    random_state=42,
    experiment_name="experiment"
):
    X = df_input[feature_cols].copy()
    y = df_input[target_col].astype(int).values

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    oof_pred_cont = np.zeros(len(df_input), dtype=float)
    oof_pred_round = np.zeros(len(df_input), dtype=int)
    fold_rows = []

    for fold, (tr, va) in enumerate(skf.split(X, y), start=1):
        X_tr, X_va = X.iloc[tr].copy(), X.iloc[va].copy()
        y_tr, y_va = y[tr], y[va]

        X_tr_t = preprocessor.fit_transform(X_tr)
        X_va_t = preprocessor.transform(X_va)

        reg = clone(base_regressor)
        reg.fit(X_tr_t, y_tr)

        va_cont = clip_predictions(reg.predict(X_va_t))
        va_round = round_predictions(va_cont)

        oof_pred_cont[va] = va_cont
        oof_pred_round[va] = va_round

        m = compute_metrics(y_va, va_round)
        fold_rows.append({
            "experiment": experiment_name,
            "fold": fold,
            "n_train": len(tr),
            "n_valid": len(va),
            **m
        })

        print(
            f"{experiment_name} | Fold {fold}: "
            f"accuracy={m['accuracy']:.4f} | mae={m['mae']:.4f} | qwk={m['qwk']:.4f}"
        )

    oof_m = compute_metrics(y, oof_pred_round)
    fold_rows.append({
        "experiment": experiment_name,
        "fold": "OOF_round",
        "n_train": None,
        "n_valid": len(df_input),
        **oof_m
    })

    metrics_table_base = pd.DataFrame(fold_rows)

    oof_detail = df_input.copy()
    oof_detail["y_true"] = y
    oof_detail["y_pred_continuous"] = oof_pred_cont
    oof_detail["y_pred_round"] = oof_pred_round

    return metrics_table_base, oof_detail


# =========================
# 9) RUN EXPERIMENT
# =========================
metrics_table_base, oof_detail = run_cv_oof_continuous(
    df_input=df_model,
    target_col=TARGET_COL,
    feature_cols=FEATURE_COLS,
    preprocessor=preprocessor,
    base_regressor=BASE_REGRESSOR,
    n_splits=N_SPLITS,
    random_state=RANDOM_STATE,
    experiment_name=EXPERIMENT_NAME_BASE
)

print("\nBase metrics table (includes OOF_round row):")
print(metrics_table_base)

y_true = oof_detail["y_true"].values
pred_cont = oof_detail["y_pred_continuous"].values
pred_round = oof_detail["y_pred_round"].values

base_metrics = compute_metrics(y_true, pred_round)
base_cm = confusion_matrix(y_true, pred_round, labels=[1, 2, 3, 4, 5])

print("\nBaseline (naive rounding) OOF metrics:")
print(base_metrics)

print("\nBaseline confusion matrix (rows=true, cols=pred):")
print(pd.DataFrame(base_cm, index=[1,2,3,4,5], columns=[1,2,3,4,5]))

best_thresholds, best_thresh_metrics = optimize_thresholds_grid(y_true, pred_cont)
pred_thresh = apply_thresholds(pred_cont, best_thresholds)
thresh_cm = confusion_matrix(y_true, pred_thresh, labels=[1, 2, 3, 4, 5])

print("\nBest optimized thresholds found:")
print(best_thresholds)

print("\nOptimized-threshold OOF metrics:")
print(best_thresh_metrics)

print("\nOptimized-threshold confusion matrix (rows=true, cols=pred):")
print(pd.DataFrame(thresh_cm, index=[1,2,3,4,5], columns=[1,2,3,4,5]))

comparison_table = pd.DataFrame([
    {
        "experiment": EXPERIMENT_NAME_BASE,
        "mapping": "naive_round",
        "thresholds": (1.5, 2.5, 3.5, 4.5),
        **base_metrics
    },
    {
        "experiment": EXPERIMENT_NAME_THRESH,
        "mapping": "optimized_thresholds",
        "thresholds": best_thresholds,
        **best_thresh_metrics
    }
])

print("\nComparison table:")
print(comparison_table)

oof_detail["y_pred_threshopt"] = pred_thresh
oof_detail["thresholds_used"] = str(best_thresholds)


# =========================
# 10) ADD TO YOUR METRICS TABLE
# =========================
metrics_table = add_experiment_result(
    metrics_table,
    experiment_name="Model L",
    model="LinearSVR",
    text_strategy="Best TFIDF + semantic extracted tags + structured categoricals",
    accuracy=best_thresh_metrics["accuracy"],
    mae=best_thresh_metrics["mae"],
    qwk=best_thresh_metrics["qwk"]
)

print("\nExperiment leaderboard:")
print(metrics_table.sort_values("QWK", ascending=False))


# =========================
# 11) OPTIONAL: INSPECT TAG DISTRIBUTIONS
# =========================
for c in SEMANTIC_TAG_COLS:
    print(f"\n{c}")
    print(df[c].value_counts(dropna=False).head(10))
    

model_L_base | Fold 1: accuracy=0.4591 | mae=0.6792 | qwk=0.4973
model_L_base | Fold 2: accuracy=0.4747 | mae=0.6835 | qwk=0.4923
model_L_base | Fold 3: accuracy=0.4747 | mae=0.6962 | qwk=0.4894
model_L_base | Fold 4: accuracy=0.4367 | mae=0.7215 | qwk=0.4945
model_L_base | Fold 5: accuracy=0.4177 | mae=0.7532 | qwk=0.4586

Base metrics table (includes OOF_round row):
     experiment       fold  n_train  n_valid  accuracy       mae       qwk
0  model_L_base          1    632.0      159  0.459119  0.679245  0.497339
1  model_L_base          2    633.0      158  0.474684  0.683544  0.492292
2  model_L_base          3    633.0      158  0.474684  0.696203  0.489422
3  model_L_base          4    633.0      158  0.436709  0.721519  0.494546
4  model_L_base          5    633.0      158  0.417722  0.753165  0.458590
5  model_L_base  OOF_round      NaN      791  0.452592  0.706700  0.486410

Baseline (naive rounding) OOF metrics:
{'accuracy': 0.4525916561314791, 'mae': 0.706700379266751, 'qwk'

In [9]:
# =========================================================
# MODEL Q
# TF-IDF + TruncatedSVD latent text embeddings
# + structured categoricals
# + LinearSVR
# + threshold optimization
# =========================================================

import numpy as np
import pandas as pd
from itertools import product

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, mean_absolute_error, cohen_kappa_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVR


# =========================
# 1) LOAD DATA
# =========================
ds_claim = pd.read_excel(path_folder + path_claim, sheet_name=0).copy()


# =========================
# 2) CONFIG
# =========================
TARGET_COL = "CAT Severity Code"
RAW_TEXT_COL = "Cause Of Injury Text"

BASE_CATEGORICAL_COLS = [
    "Division",
    "Weather Text",
    "Peril Description"
]

N_SPLITS = 5
RANDOM_STATE = 42

TFIDF_PARAMS = {
    "analyzer": "word",
    "ngram_range": (1, 4),
    "max_features": 8000,
    "min_df": 2,
    "sublinear_tf": True,
}

# Try 100, 200, 300 later if you want
SVD_N_COMPONENTS = 200

BASE_REGRESSOR = LinearSVR(
    C=1.0,
    epsilon=0.0,
    max_iter=10000,
    random_state=RANDOM_STATE
)

EXPERIMENT_NAME_BASE = "model_Q_base"
EXPERIMENT_NAME_THRESH = "model_Q_tresh"


# =========================
# 3) BASIC CLEANING
# =========================
df = ds_claim.copy()

df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
df = df[df[TARGET_COL].isin([1, 2, 3, 4, 5])].copy()
df[TARGET_COL] = df[TARGET_COL].astype(int)

for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].astype(str).str.strip()
        df.loc[df[col].isin(["nan", "None", ""]), col] = np.nan

for col in BASE_CATEGORICAL_COLS:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()
        df.loc[df[col].isin(["nan", "None", ""]), col] = np.nan

df[RAW_TEXT_COL] = df[RAW_TEXT_COL].fillna("").astype(str).str.strip()


# =========================
# 4) FEATURES
# =========================
FEATURE_COLS = [RAW_TEXT_COL] + BASE_CATEGORICAL_COLS
df_model = df[FEATURE_COLS + [TARGET_COL]].copy()


# =========================
# 5) PREPROCESSOR
# =========================
text_embedding_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(**TFIDF_PARAMS)),
    ("svd", TruncatedSVD(n_components=SVD_N_COMPONENTS, random_state=RANDOM_STATE)),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("text_embed", text_embedding_pipeline, RAW_TEXT_COL),
        ("base_cat", categorical_transformer, BASE_CATEGORICAL_COLS),
    ],
    remainder="drop"
)


# =========================
# 6) HELPERS
# =========================
def clip_predictions(pred_continuous, lo=1.0, hi=5.0):
    return np.clip(pred_continuous, lo, hi)

def round_predictions(pred_continuous):
    return np.rint(clip_predictions(pred_continuous)).astype(int)

def apply_thresholds(pred_continuous, thresholds):
    pred = clip_predictions(pred_continuous)
    t1, t2, t3, t4 = thresholds
    return np.where(
        pred < t1, 1,
        np.where(pred < t2, 2,
        np.where(pred < t3, 3,
        np.where(pred < t4, 4, 5)))
    ).astype(int)

def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "mae": mean_absolute_error(y_true, y_pred),
        "qwk": cohen_kappa_score(y_true, y_pred, weights="quadratic"),
    }

def optimize_thresholds_grid(y_true, pred_continuous):
    best = {"qwk": -np.inf, "thresholds": None, "metrics": None}

    grid_t1 = np.arange(1.2, 2.21, 0.1)
    grid_t2 = np.arange(2.0, 3.21, 0.1)
    grid_t3 = np.arange(2.8, 4.21, 0.1)
    grid_t4 = np.arange(3.6, 4.81, 0.1)

    for t1, t2, t3, t4 in product(grid_t1, grid_t2, grid_t3, grid_t4):
        if not (t1 < t2 < t3 < t4):
            continue

        y_pred = apply_thresholds(pred_continuous, (t1, t2, t3, t4))
        m = compute_metrics(y_true, y_pred)

        if m["qwk"] > best["qwk"]:
            best = {
                "qwk": m["qwk"],
                "thresholds": (float(t1), float(t2), float(t3), float(t4)),
                "metrics": m
            }

    return best["thresholds"], best["metrics"]


# =========================
# 7) CV TO GET OOF CONTINUOUS PREDICTIONS
# =========================
def run_cv_oof_continuous(
    df_input,
    target_col,
    feature_cols,
    preprocessor,
    base_regressor,
    n_splits=5,
    random_state=42,
    experiment_name="experiment"
):
    X = df_input[feature_cols].copy()
    y = df_input[target_col].astype(int).values

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    oof_pred_cont = np.zeros(len(df_input), dtype=float)
    oof_pred_round = np.zeros(len(df_input), dtype=int)
    fold_rows = []

    for fold, (tr, va) in enumerate(skf.split(X, y), start=1):
        X_tr, X_va = X.iloc[tr].copy(), X.iloc[va].copy()
        y_tr, y_va = y[tr], y[va]

        X_tr_t = preprocessor.fit_transform(X_tr)
        X_va_t = preprocessor.transform(X_va)

        reg = clone(base_regressor)
        reg.fit(X_tr_t, y_tr)

        va_cont = clip_predictions(reg.predict(X_va_t))
        va_round = round_predictions(va_cont)

        oof_pred_cont[va] = va_cont
        oof_pred_round[va] = va_round

        m = compute_metrics(y_va, va_round)
        fold_rows.append({
            "experiment": experiment_name,
            "fold": fold,
            "n_train": len(tr),
            "n_valid": len(va),
            **m
        })

        print(
            f"{experiment_name} | Fold {fold}: "
            f"accuracy={m['accuracy']:.4f} | mae={m['mae']:.4f} | qwk={m['qwk']:.4f}"
        )

    oof_m = compute_metrics(y, oof_pred_round)
    fold_rows.append({
        "experiment": experiment_name,
        "fold": "OOF_round",
        "n_train": None,
        "n_valid": len(df_input),
        **oof_m
    })

    metrics_table_base = pd.DataFrame(fold_rows)

    oof_detail = df_input.copy()
    oof_detail["y_true"] = y
    oof_detail["y_pred_continuous"] = oof_pred_cont
    oof_detail["y_pred_round"] = oof_pred_round

    return metrics_table_base, oof_detail


# =========================
# 8) RUN EXPERIMENT
# =========================
metrics_table_base, oof_detail = run_cv_oof_continuous(
    df_input=df_model,
    target_col=TARGET_COL,
    feature_cols=FEATURE_COLS,
    preprocessor=preprocessor,
    base_regressor=BASE_REGRESSOR,
    n_splits=N_SPLITS,
    random_state=RANDOM_STATE,
    experiment_name=EXPERIMENT_NAME_BASE
)

print("\nBase metrics table (includes OOF_round row):")
print(metrics_table_base)

y_true = oof_detail["y_true"].values
pred_cont = oof_detail["y_pred_continuous"].values
pred_round = oof_detail["y_pred_round"].values

base_metrics = compute_metrics(y_true, pred_round)
base_cm = confusion_matrix(y_true, pred_round, labels=[1, 2, 3, 4, 5])

print("\nBaseline (naive rounding) OOF metrics:")
print(base_metrics)

print("\nBaseline confusion matrix (rows=true, cols=pred):")
print(pd.DataFrame(base_cm, index=[1,2,3,4,5], columns=[1,2,3,4,5]))

best_thresholds, best_thresh_metrics = optimize_thresholds_grid(y_true, pred_cont)
pred_thresh = apply_thresholds(pred_cont, best_thresholds)
thresh_cm = confusion_matrix(y_true, pred_thresh, labels=[1, 2, 3, 4, 5])

print("\nBest optimized thresholds found:")
print(best_thresholds)

print("\nOptimized-threshold OOF metrics:")
print(best_thresh_metrics)

print("\nOptimized-threshold confusion matrix (rows=true, cols=pred):")
print(pd.DataFrame(thresh_cm, index=[1,2,3,4,5], columns=[1,2,3,4,5]))

comparison_table = pd.DataFrame([
    {
        "experiment": EXPERIMENT_NAME_BASE,
        "mapping": "naive_round",
        "thresholds": (1.5, 2.5, 3.5, 4.5),
        **base_metrics
    },
    {
        "experiment": EXPERIMENT_NAME_THRESH,
        "mapping": "optimized_thresholds",
        "thresholds": best_thresholds,
        **best_thresh_metrics
    }
])

print("\nComparison table:")
print(comparison_table)

oof_detail["y_pred_threshopt"] = pred_thresh
oof_detail["thresholds_used"] = str(best_thresholds)


# =========================
# 9) ADD TO YOUR METRICS TABLE
# =========================
metrics_table = add_experiment_result(
    metrics_table,
    experiment_name="Model Q",
    model="LinearSVR",
    text_strategy=f"TFIDF + SVD({SVD_N_COMPONENTS}) latent text features + structured categoricals",
    accuracy=best_thresh_metrics["accuracy"],
    mae=best_thresh_metrics["mae"],
    qwk=best_thresh_metrics["qwk"]
)

print("\nExperiment leaderboard:")
print(metrics_table.sort_values("QWK", ascending=False))


# =========================
# 10) OPTIONAL DIAGNOSTICS
# =========================
tfidf_full = TfidfVectorizer(**TFIDF_PARAMS).fit_transform(df_model[RAW_TEXT_COL].fillna(""))
svd_check = TruncatedSVD(n_components=SVD_N_COMPONENTS, random_state=RANDOM_STATE).fit(tfidf_full)

print("\nTF-IDF feature count:", tfidf_full.shape[1])
print("SVD components:", SVD_N_COMPONENTS)
print("Explained variance ratio sum:", svd_check.explained_variance_ratio_.sum())

/opt/jupyterhub/miniforge/envs/python3.12-scipy/lib/python3.12/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


model_Q_base | Fold 1: accuracy=0.3711 | mae=0.8113 | qwk=0.4326


/opt/jupyterhub/miniforge/envs/python3.12-scipy/lib/python3.12/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


model_Q_base | Fold 2: accuracy=0.4177 | mae=0.7785 | qwk=0.3809


/opt/jupyterhub/miniforge/envs/python3.12-scipy/lib/python3.12/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


model_Q_base | Fold 3: accuracy=0.4367 | mae=0.7468 | qwk=0.4558


/opt/jupyterhub/miniforge/envs/python3.12-scipy/lib/python3.12/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


model_Q_base | Fold 4: accuracy=0.3544 | mae=0.7911 | qwk=0.4759


/opt/jupyterhub/miniforge/envs/python3.12-scipy/lib/python3.12/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


model_Q_base | Fold 5: accuracy=0.3987 | mae=0.7975 | qwk=0.4391

Base metrics table (includes OOF_round row):
     experiment       fold  n_train  n_valid  accuracy       mae       qwk
0  model_Q_base          1    632.0      159  0.371069  0.811321  0.432590
1  model_Q_base          2    633.0      158  0.417722  0.778481  0.380873
2  model_Q_base          3    633.0      158  0.436709  0.746835  0.455751
3  model_Q_base          4    633.0      158  0.354430  0.791139  0.475880
4  model_Q_base          5    633.0      158  0.398734  0.797468  0.439087
5  model_Q_base  OOF_round      NaN      791  0.395702  0.785082  0.438029

Baseline (naive rounding) OOF metrics:
{'accuracy': 0.39570164348925413, 'mae': 0.7850821744627055, 'qwk': 0.4380294986050808}

Baseline confusion matrix (rows=true, cols=pred):
    1   2    3   4   5
1  10  25   22   2   0
2   2  11   49  22   6
3   0  41  192  75   7
4   0  12   48  63  33
5   1   7   47  79  37

Best optimized thresholds found:
(2.1000000000

In [10]:
# =========================================================
# MODEL R
# Best text + semantic tags + richer structured features
# + ordinal target transform + threshold optimization
#
# Main ideas:
# 1) Cause Of Injury Text -> TF-IDF
# 2) Policy Form Desc    -> separate TF-IDF
# 3) Semantic tags from Cause text
# 4) More structured columns from claims dictionary
# 5) Empirical ordinal target transform (fit on training fold only)
# 6) Threshold optimization on OOF continuous predictions
# =========================================================

import re
import numpy as np
import pandas as pd
from itertools import product

from scipy.stats import norm

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, mean_absolute_error, cohen_kappa_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVR


# =========================
# 0) OPTIONAL LOGGER HELPER
# =========================
def add_experiment_result(metrics_table, experiment_name, model, text_strategy, accuracy, mae, qwk):
    new_row = pd.DataFrame([{
        "Experiment": experiment_name,
        "Model": model,
        "TextStrategy": text_strategy,
        "Accuracy": accuracy,
        "MAE": mae,
        "QWK": qwk
    }])
    return pd.concat([metrics_table, new_row], ignore_index=True)


# =========================
# 1) LOAD DATA
# =========================
ds_claim = pd.read_excel(path_folder + path_claim, sheet_name=0).copy()


# =========================
# 2) CONFIG
# =========================
TARGET_COL = "CAT Severity Code"

CAUSE_TEXT_COL = "Cause Of Injury Text"
POLICY_TEXT_COL = "Policy Form Desc"

# Stronger structured candidates from your claims dictionary
BASE_CATEGORICAL_COLS = [
    "Division",
    "Territory",
    "Peril Description",
    "Peril Group",
    "Weather Indicator",
    "Weather Text",
    "Accident State",
    "Loss Type",
    "CAT Code",
    "Claimant Number",
]

# Derived categoricals created below
DERIVED_CATEGORICAL_COLS = [
    "zip3",
    "loss_month_cat",
    "nol_month_cat",
    "loss_dow_cat",
    "nol_dow_cat",
]

# Derived numeric created below
DERIVED_NUMERIC_COLS = [
    "report_lag_days",
]

# Optional cluster features:
# Start OFF by default. If you want, turn them on later.
USE_CLUSTER_FEATURES = False

CLUSTER_CATEGORICAL_COLS = [
    "6_cluster_cluster_id",
    "7_cluster_cluster_id",
    "8_cluster_cluster_id",
    "9_cluster_cluster_id",
    "10_cluster_cluster_id",
]

CLUSTER_NUMERIC_COLS = [
    "6_cluster_distance_to_centroid_km",
    "7_cluster_distance_to_centroid_km",
    "8_cluster_distance_to_centroid_km",
    "9_cluster_distance_to_centroid_km",
    "10_cluster_distance_to_centroid_km",
]

N_SPLITS = 5
RANDOM_STATE = 42

CAUSE_TFIDF_PARAMS = {
    "analyzer": "word",
    "ngram_range": (1, 4),
    "max_features": None,
    "min_df": 2,
    "sublinear_tf": True,
}

POLICY_TFIDF_PARAMS = {
    "analyzer": "word",
    "ngram_range": (1, 2),
    "max_features": 1000,
    "min_df": 2,
    "sublinear_tf": True,
}

BASE_REGRESSOR = LinearSVR(
    C=1.0,
    epsilon=0.0,
    max_iter=20000,
    random_state=RANDOM_STATE
)

EXPERIMENT_NAME_BASE = "model_R_base"
EXPERIMENT_NAME_THRESH = "model_R_thresh"


# =========================
# 3) BASIC CLEANING
# =========================
df = ds_claim.copy()

df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
df = df[df[TARGET_COL].isin([1, 2, 3, 4, 5])].copy()
df[TARGET_COL] = df[TARGET_COL].astype(int)

for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].astype(str).str.strip()
        df.loc[df[col].isin(["nan", "None", ""]), col] = np.nan

# Text columns
for col in [CAUSE_TEXT_COL, POLICY_TEXT_COL]:
    if col in df.columns:
        df[col] = df[col].fillna("").astype(str).str.strip()

# Categorical columns
for col in BASE_CATEGORICAL_COLS + CLUSTER_CATEGORICAL_COLS:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()
        df.loc[df[col].isin(["nan", "None", ""]), col] = np.nan

# Numeric columns
for col in CLUSTER_NUMERIC_COLS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")


# =========================
# 4) DERIVED FEATURES
# =========================
# Dates
df["Loss Date"] = pd.to_datetime(df["Loss Date"], errors="coerce")
df["NOL Date"] = pd.to_datetime(df["NOL Date"], errors="coerce")

df["report_lag_days"] = (df["NOL Date"] - df["Loss Date"]).dt.days
df["report_lag_days"] = df["report_lag_days"].clip(lower=0)

df["loss_month_cat"] = "m_" + df["Loss Date"].dt.month.fillna(-1).astype(int).astype(str)
df["nol_month_cat"] = "m_" + df["NOL Date"].dt.month.fillna(-1).astype(int).astype(str)

df["loss_dow_cat"] = "dow_" + df["Loss Date"].dt.dayofweek.fillna(-1).astype(int).astype(str)
df["nol_dow_cat"] = "dow_" + df["NOL Date"].dt.dayofweek.fillna(-1).astype(int).astype(str)

# ZIP prefix
zip_clean = df["Accident Zip"].astype(str).str.extract(r"(\d{3})", expand=False)
df["zip3"] = np.where(zip_clean.notna(), "zip3_" + zip_clean, np.nan)


# =========================
# 5) SEMANTIC TAGS (keep the simpler helpful version)
# =========================
def normalize_text(text: str) -> str:
    text = str(text).lower().strip()
    text = re.sub(r"[^a-z0-9\s\/\-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def has_any(text: str, patterns):
    return any(p in text for p in patterns)

def extract_event_family(text: str) -> str:
    if has_any(text, ["tornado", "hurricane", "storm", "wind", "gust", "shingle blown", "tree fell", "tree fall"]):
        return "wind"
    if has_any(text, ["hail", "ice dam", "ice "]):
        return "hail_ice"
    if has_any(text, ["water", "flood", "seepage", "leak", "overflow", "backup", "back-up", "sump", "sewer"]):
        return "water"
    if has_any(text, ["fire", "smoke", "burn", "char"]):
        return "fire"
    if has_any(text, ["theft", "stolen", "burglary", "robbery"]):
        return "theft"
    if has_any(text, ["collision", "crash", "struck", "hit", "rear end", "rear-end", "vehicle"]):
        return "collision_vehicle"
    if has_any(text, ["fall", "fell", "slip", "trip"]):
        return "fall_injury"
    if has_any(text, ["lightning"]):
        return "lightning"
    return "other"

def extract_damage_object(text: str) -> str:
    if has_any(text, ["roof", "shingle", "flashing", "attic"]):
        return "roof"
    if has_any(text, ["basement", "crawlspace", "crawl space", "foundation"]):
        return "basement_foundation"
    if has_any(text, ["window", "glass", "door", "garage door"]):
        return "window_door"
    if has_any(text, ["siding", "gutter", "fence", "exterior", "deck"]):
        return "exterior_structure"
    if has_any(text, ["ceiling", "wall", "floor", "drywall", "carpet", "kitchen", "bathroom", "interior"]):
        return "interior"
    if has_any(text, ["vehicle", "car", "truck", "auto"]):
        return "vehicle"
    if has_any(text, ["body", "injury", "claimant", "person", "medical"]):
        return "bodily_injury"
    return "other"

def extract_mechanism(text: str) -> str:
    if has_any(text, ["leak", "leaking", "roof leak", "flashing leak"]):
        return "leak"
    if has_any(text, ["seepage", "seep"]):
        return "seepage"
    if has_any(text, ["overflow", "backup", "back-up", "sump", "sewer"]):
        return "backup_overflow"
    if has_any(text, ["fell", "fall", "tree fell", "collapse", "collapsed"]):
        return "collapse_fall"
    if has_any(text, ["impact", "struck", "hit", "blown into"]):
        return "impact"
    if has_any(text, ["crack", "broken", "broke", "damage to"]):
        return "breakage"
    return "unknown"

def extract_weather_specific(text: str, weather_text: str, peril_desc: str) -> str:
    combo = f"{text} {weather_text} {peril_desc}".lower()
    if "tornado" in combo:
        return "tornado"
    if "hurricane" in combo:
        return "hurricane"
    if "tropical storm" in combo:
        return "tropical_storm"
    if "hail" in combo:
        return "hail"
    if "wind" in combo or "storm" in combo:
        return "wind_storm"
    if "flood" in combo:
        return "flood"
    return "none"

def extract_location_context(text: str) -> str:
    if has_any(text, ["basement", "crawlspace", "crawl space"]):
        return "basement"
    if has_any(text, ["roof", "attic", "ceiling"]):
        return "roof_upper"
    if has_any(text, ["interior", "inside", "kitchen", "bathroom", "bedroom", "living room", "wall", "floor"]):
        return "inside"
    if has_any(text, ["outside", "exterior", "yard", "driveway", "fence", "siding"]):
        return "outside"
    return "unknown"

def build_semantic_tags(dataframe: pd.DataFrame) -> pd.DataFrame:
    temp = dataframe.copy()

    norm_text = temp[CAUSE_TEXT_COL].fillna("").astype(str).map(normalize_text)
    weather_text = temp["Weather Text"].fillna("").astype(str).map(normalize_text)
    peril_desc = temp["Peril Description"].fillna("").astype(str).map(normalize_text)

    temp["tag_event_family"] = norm_text.map(extract_event_family)
    temp["tag_damage_object"] = norm_text.map(extract_damage_object)
    temp["tag_mechanism"] = norm_text.map(extract_mechanism)
    temp["tag_weather_specific"] = [
        extract_weather_specific(t, w, p) for t, w, p in zip(norm_text, weather_text, peril_desc)
    ]
    temp["tag_location_context"] = norm_text.map(extract_location_context)

    return temp

df = build_semantic_tags(df)

SEMANTIC_TAG_COLS = [
    "tag_event_family",
    "tag_damage_object",
    "tag_mechanism",
    "tag_weather_specific",
    "tag_location_context",
]


# =========================
# 6) FINAL FEATURE SETS
# =========================
CATEGORICAL_COLS = BASE_CATEGORICAL_COLS + DERIVED_CATEGORICAL_COLS + SEMANTIC_TAG_COLS
NUMERIC_COLS = DERIVED_NUMERIC_COLS

if USE_CLUSTER_FEATURES:
    CATEGORICAL_COLS = CATEGORICAL_COLS + CLUSTER_CATEGORICAL_COLS
    NUMERIC_COLS = NUMERIC_COLS + CLUSTER_NUMERIC_COLS

FEATURE_COLS = [CAUSE_TEXT_COL, POLICY_TEXT_COL] + CATEGORICAL_COLS + NUMERIC_COLS
df_model = df[FEATURE_COLS + [TARGET_COL]].copy()


# =========================
# 7) PREPROCESSOR
# =========================
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ("cause_text", TfidfVectorizer(**CAUSE_TFIDF_PARAMS), CAUSE_TEXT_COL),
        ("policy_text", TfidfVectorizer(**POLICY_TFIDF_PARAMS), POLICY_TEXT_COL),
        ("cat", categorical_transformer, CATEGORICAL_COLS),
        ("num", numeric_transformer, NUMERIC_COLS),
    ],
    remainder="drop"
)


# =========================
# 8) ORDINAL TARGET TRANSFORM
# =========================
def fit_empirical_ordinal_transform(y_train):
    """
    Build fold-specific monotone anchor values for classes 1..5
    using midpoint cumulative probabilities mapped through inverse normal.
    """
    classes = np.array([1, 2, 3, 4, 5])
    counts = pd.Series(y_train).value_counts().reindex(classes, fill_value=0).values.astype(float)

    # small smoothing so no class gets p=0
    counts = counts + 0.5
    probs = counts / counts.sum()

    cum_prev = 0.0
    anchor_map = {}

    for cls, p in zip(classes, probs):
        midpoint = cum_prev + p / 2.0
        midpoint = np.clip(midpoint, 1e-6, 1 - 1e-6)
        anchor_map[int(cls)] = float(norm.ppf(midpoint))
        cum_prev += p

    return anchor_map

def apply_target_transform(y, anchor_map):
    return np.array([anchor_map[int(v)] for v in y], dtype=float)

def nearest_anchor_class(pred_scores, anchor_map):
    classes = np.array(sorted(anchor_map.keys()))
    anchors = np.array([anchor_map[c] for c in classes])

    out = []
    for s in pred_scores:
        idx = np.argmin(np.abs(anchors - s))
        out.append(int(classes[idx]))
    return np.array(out, dtype=int)


# =========================
# 9) HELPERS
# =========================
def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "mae": mean_absolute_error(y_true, y_pred),
        "qwk": cohen_kappa_score(y_true, y_pred, weights="quadratic"),
    }

def apply_thresholds(pred_scores, thresholds):
    t1, t2, t3, t4 = thresholds
    return np.where(
        pred_scores < t1, 1,
        np.where(pred_scores < t2, 2,
        np.where(pred_scores < t3, 3,
        np.where(pred_scores < t4, 4, 5)))
    ).astype(int)

def optimize_thresholds_grid(y_true, pred_scores):
    best = {"qwk": -np.inf, "thresholds": None, "metrics": None}

    # Wider grids because transformed target is not on 1..5 scale
    grid_t1 = np.arange(-2.0, 0.01, 0.1)
    grid_t2 = np.arange(-1.0, 1.01, 0.1)
    grid_t3 = np.arange(0.0, 2.01, 0.1)
    grid_t4 = np.arange(0.5, 3.01, 0.1)

    for t1, t2, t3, t4 in product(grid_t1, grid_t2, grid_t3, grid_t4):
        if not (t1 < t2 < t3 < t4):
            continue

        y_pred = apply_thresholds(pred_scores, (t1, t2, t3, t4))
        m = compute_metrics(y_true, y_pred)

        if m["qwk"] > best["qwk"]:
            best = {
                "qwk": m["qwk"],
                "thresholds": (float(t1), float(t2), float(t3), float(t4)),
                "metrics": m
            }

    return best["thresholds"], best["metrics"]


# =========================
# 10) CV TO GET OOF CONTINUOUS PREDICTIONS
# =========================
def run_cv_oof_continuous_transformed(
    df_input,
    target_col,
    feature_cols,
    preprocessor,
    base_regressor,
    n_splits=5,
    random_state=42,
    experiment_name="experiment"
):
    X = df_input[feature_cols].copy()
    y_true_orig = df_input[target_col].astype(int).values

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    oof_pred_cont = np.zeros(len(df_input), dtype=float)
    oof_pred_round = np.zeros(len(df_input), dtype=int)
    fold_rows = []

    for fold, (tr, va) in enumerate(skf.split(X, y_true_orig), start=1):
        X_tr, X_va = X.iloc[tr].copy(), X.iloc[va].copy()
        y_tr_orig, y_va_orig = y_true_orig[tr], y_true_orig[va]

        # fold-specific ordinal transform
        anchor_map = fit_empirical_ordinal_transform(y_tr_orig)
        y_tr_trans = apply_target_transform(y_tr_orig, anchor_map)

        X_tr_t = preprocessor.fit_transform(X_tr)
        X_va_t = preprocessor.transform(X_va)

        reg = clone(base_regressor)
        reg.fit(X_tr_t, y_tr_trans)

        va_cont = reg.predict(X_va_t)
        va_round = nearest_anchor_class(va_cont, anchor_map)

        oof_pred_cont[va] = va_cont
        oof_pred_round[va] = va_round

        m = compute_metrics(y_va_orig, va_round)
        fold_rows.append({
            "experiment": experiment_name,
            "fold": fold,
            "n_train": len(tr),
            "n_valid": len(va),
            **m
        })

        print(
            f"{experiment_name} | Fold {fold}: "
            f"accuracy={m['accuracy']:.4f} | "
            f"mae={m['mae']:.4f} | "
            f"qwk={m['qwk']:.4f}"
        )

    oof_m = compute_metrics(y_true_orig, oof_pred_round)
    fold_rows.append({
        "experiment": experiment_name,
        "fold": "OOF_round",
        "n_train": None,
        "n_valid": len(df_input),
        **oof_m
    })

    metrics_table_base = pd.DataFrame(fold_rows)

    oof_detail = df_input.copy()
    oof_detail["y_true"] = y_true_orig
    oof_detail["y_pred_continuous"] = oof_pred_cont
    oof_detail["y_pred_round"] = oof_pred_round

    return metrics_table_base, oof_detail


# =========================
# 11) RUN EXPERIMENT
# =========================
metrics_table_base, oof_detail = run_cv_oof_continuous_transformed(
    df_input=df_model,
    target_col=TARGET_COL,
    feature_cols=FEATURE_COLS,
    preprocessor=preprocessor,
    base_regressor=BASE_REGRESSOR,
    n_splits=N_SPLITS,
    random_state=RANDOM_STATE,
    experiment_name=EXPERIMENT_NAME_BASE
)

print("\nBase metrics table (includes OOF_round row):")
print(metrics_table_base)

y_true = oof_detail["y_true"].values
pred_cont = oof_detail["y_pred_continuous"].values
pred_round = oof_detail["y_pred_round"].values

base_metrics = compute_metrics(y_true, pred_round)
base_cm = confusion_matrix(y_true, pred_round, labels=[1, 2, 3, 4, 5])

print("\nBaseline (nearest-anchor mapping) OOF metrics:")
print(base_metrics)

print("\nBaseline confusion matrix (rows=true, cols=pred):")
print(pd.DataFrame(base_cm, index=[1,2,3,4,5], columns=[1,2,3,4,5]))

best_thresholds, best_thresh_metrics = optimize_thresholds_grid(y_true, pred_cont)
pred_thresh = apply_thresholds(pred_cont, best_thresholds)
thresh_cm = confusion_matrix(y_true, pred_thresh, labels=[1, 2, 3, 4, 5])

print("\nBest optimized thresholds found:")
print(best_thresholds)

print("\nOptimized-threshold OOF metrics:")
print(best_thresh_metrics)

print("\nOptimized-threshold confusion matrix (rows=true, cols=pred):")
print(pd.DataFrame(thresh_cm, index=[1,2,3,4,5], columns=[1,2,3,4,5]))

comparison_table = pd.DataFrame([
    {
        "experiment": EXPERIMENT_NAME_BASE,
        "mapping": "nearest_anchor",
        "thresholds": "fold_specific_anchor_map",
        **base_metrics
    },
    {
        "experiment": EXPERIMENT_NAME_THRESH,
        "mapping": "optimized_thresholds",
        "thresholds": best_thresholds,
        **best_thresh_metrics
    }
])

print("\nComparison table:")
print(comparison_table)

oof_detail["y_pred_threshopt"] = pred_thresh
oof_detail["thresholds_used"] = str(best_thresholds)


# =========================
# 12) ADD TO LEADERBOARD
# =========================
metrics_table = add_experiment_result(
    metrics_table,
    experiment_name="Model R",
    model="LinearSVR",
    text_strategy="Cause TFIDF + Policy TFIDF + semantic tags + richer structured vars + ordinal target transform",
    accuracy=best_thresh_metrics["accuracy"],
    mae=best_thresh_metrics["mae"],
    qwk=best_thresh_metrics["qwk"]
)

print("\nExperiment leaderboard:")
print(metrics_table.sort_values("QWK", ascending=False))


# =========================
# 13) QUICK DIAGNOSTICS
# =========================
print("\nTarget distribution:")
print(df_model[TARGET_COL].value_counts().sort_index())

print("\nFeature overview:")
print("Cause text column:", CAUSE_TEXT_COL)
print("Policy text column:", POLICY_TEXT_COL)
print("Categorical count:", len(CATEGORICAL_COLS))
print("Numeric count:", len(NUMERIC_COLS))
print("Cluster features active:", USE_CLUSTER_FEATURES)

model_R_base | Fold 1: accuracy=0.3962 | mae=0.7610 | qwk=0.4902
model_R_base | Fold 2: accuracy=0.4304 | mae=0.7848 | qwk=0.3901
model_R_base | Fold 3: accuracy=0.4304 | mae=0.7911 | qwk=0.4246
model_R_base | Fold 4: accuracy=0.4051 | mae=0.7595 | qwk=0.5121
model_R_base | Fold 5: accuracy=0.4177 | mae=0.7532 | qwk=0.4432

Base metrics table (includes OOF_round row):
     experiment       fold  n_train  n_valid  accuracy       mae       qwk
0  model_R_base          1    632.0      159  0.396226  0.761006  0.490246
1  model_R_base          2    633.0      158  0.430380  0.784810  0.390101
2  model_R_base          3    633.0      158  0.430380  0.791139  0.424644
3  model_R_base          4    633.0      158  0.405063  0.759494  0.512121
4  model_R_base          5    633.0      158  0.417722  0.753165  0.443202
5  model_R_base  OOF_round      NaN      791  0.415929  0.769912  0.452871

Baseline (nearest-anchor mapping) OOF metrics:
{'accuracy': 0.415929203539823, 'mae': 0.769911504424778